In [27]:
import numpy as np
import pandas as pd
import featuretools as ft
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
from woodwork.logical_types import Categorical

In [28]:
def loadRecordings(datasets):
    sessions = []
    data = []
    for entry in datasets:
        df = pd.read_csv(entry["path"])
        sessionId = f"{entry['driver']}_{entry['run']}"
        
        df = df.apply(pd.to_numeric, errors="coerce").drop([0,1]).interpolate().dropna(axis=1, how="all")   # Dropping row 0 (Text) and row 1 (nan / zero values)
        df = df.loc[:, (df != df.iloc[0]).any()]
        df["sessionId"] = sessionId
        
        data.append(df)
        sessions.append({
            "sessionId": sessionId,
            "driver": entry["driver"],
            "run": entry["run"]
        })
    return pd.DataFrame(sessions), pd.concat(data, ignore_index=True)

In [29]:
def expandFeatures(sessions: pd.DataFrame, data: pd.DataFrame) -> tuple[pd.DataFrame, list]:
    es = ft.EntitySet(id="driver_analysis")

    es = es.add_dataframe(
        dataframe_name="sessions",
        dataframe=sessions,
        index="sessionId",
        logical_types={
            "driver": Categorical,
            "run": Categorical
        }
    )

    es = es.add_dataframe(
        dataframe_name="telemetry",
        dataframe=data,
        index="telemetryId",
        make_index=True,
        time_index="timestamp"
    )

    es = es.add_relationship(
        parent_dataframe_name="sessions",
        parent_column_name="sessionId",
        child_dataframe_name="telemetry",
        child_column_name="sessionId"
    )

    featureMatrix, featureDefs = ft.dfs(
        entityset=es,
        target_dataframe_name="sessions",
        ignore_columns={
            "telemetry": ["driver", "run"]
        },
        agg_primitives=["mean", "std", "min", "max", "skew", "count"],
        trans_primitives=["diff", "absolute"],
        max_depth=2,
        verbose=True
    )

    return featureMatrix, featureDefs


In [30]:
datasets = [
    { "path": "./recordings/recording_fabian_1.csv", "driver": 0, "run": 0 },
    { "path": "./recordings/recording_fabian_2.csv", "driver": 0, "run": 1 },
    #{ "path": "./recordings/recording_florian_2.csv", "driver": 1, "run": 0 },
    #{ "path": "./recordings/recording_florian_3.csv", "driver": 1, "run": 1 },
    { "path": "./recordings/recording_matthias_2.csv", "driver": 2, "run": 0 },
    { "path": "./recordings/recording_matthias_3.csv", "driver": 2, "run": 1 }   
]   

sessions, data = loadRecordings(datasets)
aggMatrix, aggDef = expandFeatures(sessions, data)

aggMatrix = aggMatrix.dropna(axis=1)

C:\Users\games\AppData\Local\Temp\ipykernel_6064\170129305.py:5: DtypeWarning: Columns (0,1,6,7,8,13,16,20,24,28,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,64,65,66,67,68,69,70,71,72,73) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(entry["path"])
C:\Users\games\AppData\Local\Temp\ipykernel_6064\170129305.py:5: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,18,21,25,29,33,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,69,70,71,72,73,78) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(entry["path"])
C:\Users\games\AppData\Local\Temp\ipykernel_6064\170129305.py:5: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,20,23,27,31,35,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,71,72,73,78) have mixed types. Specify dtype option on import or set low_memory=Fals

Built 1005 features
Elapsed: 00:00 | Progress:  27%|██▋       

p:\Apps\Entwicklung\Software\Anaconda\envs\MSuT\Lib\site-packages\featuretools\computational_backends\feature_set_calculator.py:785: FutureWarning: The provided callable <function std at 0x0000022056EA2980> is currently using SeriesGroupBy.std. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "std" instead.
  ).agg(to_agg)
p:\Apps\Entwicklung\Software\Anaconda\envs\MSuT\Lib\site-packages\featuretools\computational_backends\feature_set_calculator.py:785: FutureWarning: The provided callable <function max at 0x0000022056EA1E40> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  ).agg(to_agg)
p:\Apps\Entwicklung\Software\Anaconda\envs\MSuT\Lib\site-packages\featuretools\computational_backends\feature_set_calculator.py:785: FutureWarning: The provided callable <function min at 0x0000022056EA1F80> is curr

Elapsed: 00:00 | Progress:  95%|█████████▍

p:\Apps\Entwicklung\Software\Anaconda\envs\MSuT\Lib\site-packages\featuretools\computational_backends\feature_set_calculator.py:818: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  frame = frame.fillna(fillna_dict)


Elapsed: 00:01 | Progress: 100%|██████████


In [31]:
def scoreFeatures(
    data: pd.DataFrame,
    driverCol: str = "driver",
    runCol: str = "run",
    eps: float = 1e-6
) -> pd.DataFrame:
    """
    Berechnet für jedes Feature einen Discriminability-Score:
    - hohe Inter-Driver-Varianz
    - niedrige Intra-Driver-Varianz

    Rückgabe: DataFrame mit Score und Teilmetriken
    """

    feature_cols = [
        c for c in data.columns
        if c not in [driverCol, runCol]
    ]

    results = []

    for feature in feature_cols:
        values = data[[driverCol, runCol, feature]].dropna()

        if values.empty:
            continue

        # Intra-Driver-Streuung
        intra_stds = []
        driver_means = []

        for driver, g in values.groupby(driverCol):
            if len(g) < 2:
                continue  # kein intra-Vergleich möglich

            intra_stds.append(g[feature].std())
            driver_means.append(g[feature].mean())

        if len(intra_stds) == 0 or len(driver_means) < 2:
            continue

        intra_var = np.mean(intra_stds)
        inter_var = np.std(driver_means)

        score = inter_var / (inter_var + intra_var + eps)

        results.append({
            "feature": feature,
            "score": score,
            "inter_driver_std": inter_var,
            "intra_driver_std": intra_var
        })

    return (
        pd.DataFrame(results)
        .sort_values("score", ascending=False)
        .reset_index(drop=True)
    )

featureScores = scoreFeatures(aggMatrix)
featureScores

C:\Users\games\AppData\Local\Temp\ipykernel_6064\3604711138.py:32: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for driver, g in values.groupby(driverCol):
C:\Users\games\AppData\Local\Temp\ipykernel_6064\3604711138.py:32: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for driver, g in values.groupby(driverCol):
C:\Users\games\AppData\Local\Temp\ipykernel_6064\3604711138.py:32: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

,feature,score,inter_driver_std,intra_driver_std
0,MIN(telemetry.DIFF(car0_wheel0_skid_factor_lat)),0.839613,0.006072,0.001159
1,SKEW(telemetry.car0_engine_load),0.821047,0.150554,0.032813
2,ABSOLUTE(SKEW(telemetry.car0_engine_load)),0.821047,0.150554,0.032813
3,SKEW(telemetry.ABSOLUTE(car0_engine_load)),0.821047,0.150554,0.032813
4,SKEW(telemetry.throttle),0.790132,0.270971,0.071972
...,...,...,...,...
557,MIN(telemetry.DIFF(timestamp)),0.000000,0.000000,0.000000
558,ABSOLUTE(MAX(telemetry.car0{1}.shift_up)),0.000000,0.000000,0.000000
559,ABSOLUTE(MIN(telemetry.throttle)),0.000000,0.000000,0.000000
560,ABSOLUTE(MAX(telemetry.car0{1}.shift_down)),0.000000,0.000000,0.000000


In [33]:
def selectTopKFeatures(featureScores: pd.DataFrame, k: int = 5):
    return featureScores.head(k)["feature"].tolist()

def identifyDrivers(
    data: pd.DataFrame,
    featureScores: pd.DataFrame,
    driverCol: str = "driver",
    runCol: str = "run",
    k: int = 5
):

    features = selectTopKFeatures(featureScores, k)

    X = data[features].fillna(0).values # Train-Data Features
    y = data[driverCol].values          # Train-Data Results
    runs = data[runCol].values          

    results = []

    for testRun in np.unique(runs):     # Schleife läuft n Mal, wobei n die Anzahl der Durchläufe pro Fahrer ist (hier: 2x)
        trainMask = runs != testRun     # Masken für Test-Train Split   --> [False, True, False, True]
        testMask = runs == testRun      #                               --> [True, False, True, False]

        XTrain, XTest = X[trainMask], X[testMask]
        yTrain, yTest = y[trainMask], y[testMask]

        # Skalierung
        scaler = StandardScaler()
        XTrain = scaler.fit_transform(XTrain)
        XTest = scaler.transform(XTest)

        # Zentroiden
        centroids = {
            driver: XTrain[yTrain == driver].mean(axis=0)
            for driver in np.unique(yTrain)
        }

        # Klassifikation per nächstem Zentroid
        yPred = []
        for x in XTest:
            distances = {
                driver: np.linalg.norm(x - centroid)
                for driver, centroid in centroids.items()
            }
            yPred.append(min(distances, key=distances.get))

        acc = accuracy_score(yTest, yPred)
        cm = confusion_matrix(yTest, yPred)

        results.append({
            "testRun": testRun,
            "accuracy": acc,
            "confusionMatrix": cm
        })

    return results

results = identifyDrivers(
    data=aggMatrix,
    featureScores=featureScores,
    k=5
)

for r in results:
    print(f"Test-Run {r['testRun']}: Accuracy = {r['accuracy']}")
    print(r["confusionMatrix"])

Test-Run 0: Accuracy = 1.0
[[1 0]
 [0 1]]
Test-Run 1: Accuracy = 1.0
[[1 0]
 [0 1]]


In [39]:
def identifySamples(
    dataPaths: list,
    samplePaths: list,
    featureScores: pd.DataFrame,
    k: int = 5
):
    dataPaths.extend(
        {"path": samplePath, "driver": -1, "run": 0}
        for samplePath in samplePaths
    )
    sessions, data = loadRecordings(dataPaths)
    aggMatrix, aggDef = expandFeatures(sessions, data)
    aggMatrix = aggMatrix.dropna(axis=1)


    #sampleSessions, sampleDf = loadRecordings([{"path": samplePath, driverCol: -1, runCol: 0}])
    #sample, sampleDef = expandFeatures(sampleSessions, sampleDf)
    #sample = sample.dropna(axis=1)

    #sample.to_csv("./preprocessed/SampleExpansion.csv")
    samples = aggMatrix[aggMatrix["driver"] == -1]
    features = featureScores.head(k)["feature"].tolist()

    XTrain = aggMatrix.loc[aggMatrix["driver"] != -1, features].fillna(0).values    # Nur k beste Features und bekannte Driver in den Trainingsdatensatz einfließen lassen
    yTrain = aggMatrix.loc[aggMatrix["driver"] != -1, "driver"].fillna(0).values    # Bekannte Driver (Y) des Trainingsdatensatzes auswählen
    XSamples = samples[features].fillna(0).values

    print(XTrain)
    print(yTrain)
    print(XSamples)

    # Skalierung
    scaler = StandardScaler()
    XTrain = scaler.fit_transform(XTrain)
    XSamples = scaler.transform(XSamples)

    print(XTrain)
    print(XSamples)

    # Zentroiden
    centroids = {
        driver: XTrain[yTrain == driver].mean(axis=0)
        for driver in np.unique(yTrain)
        if driver != -1                                 # Die gesuchten Fahrer für die Klassifikation aus dem Trainingsdatensatz herausfiltern. Wäre auch früher schon möglich
    }

    # Klassifikation per nächstem Zentroid
    distances = {
        driver: np.linalg.norm(XSamples - centroid, axis=1)
        for driver, centroid in centroids.items()
    }

    predictedDrivers = [
        min(distances, key=lambda d: distances[d][i])
        for i in range(XSamples.shape[0])
    ]

    return samples, predictedDrivers, distances

In [46]:
datasets = [
    { "path": "./recordings/recording_fabian_1.csv", "driver": 0, "run": 0 },
    { "path": "./recordings/recording_fabian_2.csv", "driver": 0, "run": 1 },
    #{ "path": "./recordings/recording_florian_2.csv", "driver": 1, "run": 0 },
    #{ "path": "./recordings/recording_florian_3.csv", "driver": 1, "run": 1 },
    #{ "path": "./recordings/recording_matthias_2.csv", "driver": 2, "run": 0 },
    { "path": "./recordings/recording_matthias_3.csv", "driver": 2, "run": 1}  
]
samples = [
    "./recordings/recording_matthias_2.csv"
]

sample, predictions, distances = identifySamples(datasets, samples, featureScores)

C:\Users\games\AppData\Local\Temp\ipykernel_6064\170129305.py:5: DtypeWarning: Columns (0,1,6,7,8,13,16,20,24,28,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,64,65,66,67,68,69,70,71,72,73) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(entry["path"])
C:\Users\games\AppData\Local\Temp\ipykernel_6064\170129305.py:5: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,18,21,25,29,33,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,69,70,71,72,73,78) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(entry["path"])
C:\Users\games\AppData\Local\Temp\ipykernel_6064\170129305.py:5: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,24,27,31,35,39,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,75,76,77,78) have mixed types. Specify dtype option on import or set low_memory=Fals

Built 1005 features
Elapsed: 00:00 | Progress:  27%|██▋       

p:\Apps\Entwicklung\Software\Anaconda\envs\MSuT\Lib\site-packages\featuretools\computational_backends\feature_set_calculator.py:785: FutureWarning: The provided callable <function std at 0x0000022056EA2980> is currently using SeriesGroupBy.std. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "std" instead.
  ).agg(to_agg)
p:\Apps\Entwicklung\Software\Anaconda\envs\MSuT\Lib\site-packages\featuretools\computational_backends\feature_set_calculator.py:785: FutureWarning: The provided callable <function max at 0x0000022056EA1E40> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  ).agg(to_agg)
p:\Apps\Entwicklung\Software\Anaconda\envs\MSuT\Lib\site-packages\featuretools\computational_backends\feature_set_calculator.py:785: FutureWarning: The provided callable <function min at 0x0000022056EA1F80> is curr

Elapsed: 00:00 | Progress:  95%|█████████▍

p:\Apps\Entwicklung\Software\Anaconda\envs\MSuT\Lib\site-packages\featuretools\computational_backends\feature_set_calculator.py:818: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  frame = frame.fillna(fillna_dict)


Elapsed: 00:01 | Progress: 100%|██████████
[[-0.0323153   1.6348938   1.6348938   1.6348938   1.43153132]
 [-0.03512664  1.60050814  1.60050814  1.60050814  1.40924597]
 [-0.0460993   1.28738085  1.28738085  1.28738085  0.7878057 ]]
[0, 0, 2]
Categories (3, int64): [-1, 0, 2]
[[-0.04563252  1.34580536  1.34580536  1.34580536  0.9690877 ]]
[[ 0.93017964  0.81421739  0.81421739  0.81421739  0.74412669]
 [ 0.4574471   0.5942841   0.5942841   0.5942841   0.66942914]
 [-1.38762674 -1.40850149 -1.40850149 -1.40850149 -1.41355583]]
[[-1.30913711 -1.03481393 -1.03481393 -1.03481393 -0.80592273]]


In [47]:
predictions

[np.int64(2)]